# Taller 4 NetFlow v9 y saturación WAN

Este notebook analiza directamente el PCAP sintético del taller mediante Python estándar. No requiere Scapy, tshark ni un colector NetFlow. Después enriquece los registros con el CSV y construye evidencia cuantitativa del incidente.

**Caso:** detectar qué conversación llevó un enlace WAN de 20 Mbps a una condición de saturación entre 10:00 y 10:18.


## Participación de Colab

Wireshark permite inspeccionar paquetes, templates y campos individuales. Colab permite procesar todos los registros, convertir octetos a GB, calcular tasas, agrupar top talkers, comparar fases y aplicar una detección simple de anomalías. Ambas vistas deben producir la misma evidencia.


In [ ]:
from pathlib import Path
import base64, io, os, socket, struct
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

LINK_MBPS = 20.0
print('Entorno preparado. Capacidad del enlace:', LINK_MBPS, 'Mbps')


## 1 Generar automáticamente los archivos del laboratorio

    Este notebook es autocontenido. La siguiente celda reconstruye `netflow_wan_congestion_v9.pcap` y `netflow_records.csv` sin pedir archivos externos. Después podrá descargarlos para abrir el PCAP en Wireshark.


In [ ]:
PCAP = 'netflow_wan_congestion_v9.pcap'
CSV = 'netflow_records.csv'
PCAP_B64 = '1MOyoQIABAAAAAAAAAAAAP//AAABAAAAwZDPagAAAAB6AgAAegIAAAARIjNEVQBmVUQzIggARQACbAPpAABAEV5oCv8AAQr/ADInCwgHAlgAAAAJABAAhNPoas+QwQAAAAAAAAAqAAAAOAEAAAwACAAEAAwABAAHAAIACwACAAQAAQAGAAEAAQAEAAIABAAKAAIADgACABYABAAVAAQBAAIECgoeGawUChTHOAG7BhsCgN6AAACYWAAMAAMAbd0AAHJw4AoKKAjGM2QoyyANlhEAA3UCgAAAv2gADAADAG3dAABycOAKCh4sywBxUM8IAbsGGwGb/MAAAFnYAAwAAwBt3QAAcnDgCgoeGQoKATXS8AA1EQAAIZHAAAAhNAAMAAgAbd0AAHJw4AoKMjzGM2Ra1tgBuwYbAPQkAAAANrAADAADAG3dAABycOAKCh4ZrBQKFMc5AbsGGwKfYwAAAJ0IAAwAAwB3BMAAe5igCgooCMYzZCjLIQ2WEQADoslAAADFRAAMAAMAdwTAAHuYoAoKHizLAHFQzwkBuwYbAbLgIAAAXVwADAADAHcEwAB7mKAKCh4ZCgoBNdLxADURAAAjGGAAACH8AAwACAB3BMAAe5igCgoyPMYzZFrW2QG7BhsBA2ZAAAA5CAAMAAMAdwTAAHuYoAoKHhmsFAoUxzoBuwYbAr3ngAAAobgADAADAIAsgACEwGAKCigIxjNkKMsiDZYRAAPQkAAAAMsgAAwAAwCALIAAhMBgCgoeLMsAcVDPCgG7BhsBycOAAABg4AAMAAMAgCyAAITAYAoKHhkKCgE10vIANREAACSfAAAAIsQADAAIAIAsgACEwGAKCjI8xjNkWtbaAbsGGwESqIAAADtgAAwAAwCALIAAhMBgAAAtnc9qAAAAAGoBAABqAQAAABEiM0RVAGZVRDMiCABFAAFcA+oAAEARX3cK/wABCv8AMicLCAcBSAAAAAkACAC1Wchqz50tAAAADwAAACoAAAA4AQAADAAIAAQADAAEAAcAAgALAAIABAABAAYAAQABAAQAAgAEAAoAAgAOAAIAFgAEABUABAEAAPQKChQPrBQFHsuwAbsGG2tJ0gAAEuvAAAwAAwCky4AAtUZACgoeGawUChTHpgG7BhsGjneAAAF3AAAMAAMApMuAALVGQAoKKAjGM2Qoy44NlhEADRzvAAACzsAADAADAKTLgAC1RkAKCh4sywBxUM92AbsGGwhYOwAAAdDYAAwAAwCky4AAtUZACgoyPMYzZFrXRgG7BhsR4aMAAAPYYAAMAAMApMuAALVGQAoKHhkKCgE1014ANREAAExLQAAARlAADAAIAKTLgAC1RkAKCmMKrBQKFNsuABYGGwCYloAAACkEAAwAAwCky4AAtUZAAAApoc9qAAAAAM4BAADOAQAAABEiM0RVAGZVRDMiCABFAAHAA+sAAEARXxIK/wABCv8AMicLCAcBrAAAAAkACwDE6ihqz6EpAAAAFgAAACoAAAA4AQAADAAIAAQADAAEAAcAAgALAAIABAABAAYAAQABAAQAAgAEAAoAAgAOAAIAFgAEABUABAEAAVgKCh4ZrBQKFMgAAbsGGwLcbAAAAKvgAAwAAwC3GwAAu67gCgooCMYzZCjL6A2WEQADsguAAADLIAAMAAMAtxsAALuu4AoKHizLAHFQz9ABuwYbAdkFwAAAZZAADAADALcbAAC7ruAKCh4ZCgoBNdO4ADURAAAmJaAAACPwAAwACAC3GwAAu67gCgoyPMYzZFrXoAG7BhsBEqiAAAA6mAAMAAMAtxsAALuu4AoKHhmsFAoUyAEBuwYbAvrwgAAAr8gADAADAMBCwADE1qAKCigIxjNkKMvpDZYRAAPQkAAAAM8IAAwAAwDAQsAAxNagCgoeLMsAcVDP0QG7BhsB6EgAAABoTAAMAAMAwELAAMTWoAoKHhkKCgE107kANREAACesQAAAJLgADAAIAMBCwADE1qAKCjI8xjNkWtehAbsGGwEh6sAAADyMAAwAAwDAQsAAxNag'
CSV_B64 = 'Zmxvd19pZCxwaGFzZSxzcmNfaXAsc3JjX25hbWUsZHN0X2lwLGRzdF9uYW1lLHNyY19wb3J0LGRzdF9wb3J0LHByb3RvY29sLHByb3RvY29sX25hbWUsdGNwX2ZsYWdzX2hleCxieXRlcyxwYWNrZXRzLGlucHV0X2lmLG91dHB1dF9pZixzdGFydF90aW1lLGVuZF90aW1lLGR1cmF0aW9uX3MsYXZnX21icHMsbGlua191dGlsaXphdGlvbl9wY3QsYXBwbGljYXRpb24sZXhwZWN0ZWRfcm9sZQ0KRjAwMSxMSU5FQV9CQVNFLDEwLjEwLjMwLjI1LFVTRVItMDI1LDE3Mi4yMC4xMC4yMCxBUEktUFJPRCw1MTAwMCw0NDMsNixUQ1AsMHgxYiw0MjAwMDAwMCwzOTAwMCwxMiwzLDIwMjYtMTAtMTRUMDk6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOTowNTowMC0wNTowMCwzMDAsMS4xMiw1LjYsQVBJIEhUVFBTLFRyw6FmaWNvIGRlIG5lZ29jaW8NCkYwMDIsTElORUFfQkFTRSwxMC4xMC40MC44LFVDLUNMSUVOVC0wOCwxOTguNTEuMTAwLjQwLENPTExBQi1DTE9VRCw1MjAwMCwzNDc4LDE3LFVEUCwweDAwLDU4MDAwMDAwLDQ5MDAwLDEyLDMsMjAyNi0xMC0xNFQwOTowMDowMC0wNTowMCwyMDI2LTEwLTE0VDA5OjA1OjAwLTA1OjAwLDMwMCwxLjU0NjY2Nyw3LjczMzMsQ29sYWJvcmFjacOzbiBVRFAsQ29tdW5pY2FjaW9uZXMNCkYwMDMsTElORUFfQkFTRSwxMC4xMC4zMC40NCxVU0VSLTA0NCwyMDMuMC4xMTMuODAsV0VCLVNBQVMsNTMwMDAsNDQzLDYsVENQLDB4MWIsMjcwMDAwMDAsMjMwMDAsMTIsMywyMDI2LTEwLTE0VDA5OjAwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MDU6MDAtMDU6MDAsMzAwLDAuNzIsMy42LE5hdmVnYWNpw7NuIEhUVFBTLFRyw6FmaWNvIGRlIHVzdWFyaW8NCkYwMDQsTElORUFfQkFTRSwxMC4xMC4zMC4yNSxVU0VSLTAyNSwxMC4xMC4xLjUzLEROUy0wMSw1NDAwMCw1MywxNyxVRFAsMHgwMCwyMjAwMDAwLDg1MDAsMTIsOCwyMDI2LTEwLTE0VDA5OjAwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MDU6MDAtMDU6MDAsMzAwLDAuMDU4NjY3LDAuMjkzMyxETlMsSW5mcmFlc3RydWN0dXJhDQpGMDA1LExJTkVBX0JBU0UsMTAuMTAuNTAuNjAsV1NVUy1DTElFTlQtNjAsMTk4LjUxLjEwMC45MCxVUERBVEUtQ0ROLDU1MDAwLDQ0Myw2LFRDUCwweDFiLDE2MDAwMDAwLDE0MDAwLDEyLDMsMjAyNi0xMC0xNFQwOTowMDowMC0wNTowMCwyMDI2LTEwLTE0VDA5OjA1OjAwLTA1OjAwLDMwMCwwLjQyNjY2NywyLjEzMzMsQWN0dWFsaXphY2lvbmVzLE1hbnRlbmltaWVudG8NCkYwMDYsTElORUFfQkFTRSwxMC4xMC4zMC4yNSxVU0VSLTAyNSwxNzIuMjAuMTAuMjAsQVBJLVBST0QsNTEwMDEsNDQzLDYsVENQLDB4MWIsNDQwMDAwMDAsNDAyMDAsMTIsMywyMDI2LTEwLTE0VDA5OjEwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MTU6MDAtMDU6MDAsMzAwLDEuMTczMzMzLDUuODY2NyxBUEkgSFRUUFMsVHLDoWZpY28gZGUgbmVnb2Npbw0KRjAwNyxMSU5FQV9CQVNFLDEwLjEwLjQwLjgsVUMtQ0xJRU5ULTA4LDE5OC41MS4xMDAuNDAsQ09MTEFCLUNMT1VELDUyMDAxLDM0NzgsMTcsVURQLDB4MDAsNjEwMDAwMDAsNTA1MDAsMTIsMywyMDI2LTEwLTE0VDA5OjEwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MTU6MDAtMDU6MDAsMzAwLDEuNjI2NjY3LDguMTMzMyxDb2xhYm9yYWNpw7NuIFVEUCxDb211bmljYWNpb25lcw0KRjAwOCxMSU5FQV9CQVNFLDEwLjEwLjMwLjQ0LFVTRVItMDQ0LDIwMy4wLjExMy44MCxXRUItU0FBUyw1MzAwMSw0NDMsNixUQ1AsMHgxYiwyODUwMDAwMCwyMzkwMCwxMiwzLDIwMjYtMTAtMTRUMDk6MTA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOToxNTowMC0wNTowMCwzMDAsMC43NiwzLjgsTmF2ZWdhY2nDs24gSFRUUFMsVHLDoWZpY28gZGUgdXN1YXJpbw0KRjAwOSxMSU5FQV9CQVNFLDEwLjEwLjMwLjI1LFVTRVItMDI1LDEwLjEwLjEuNTMsRE5TLTAxLDU0MDAxLDUzLDE3LFVEUCwweDAwLDIzMDAwMDAsODcwMCwxMiw4LDIwMjYtMTAtMTRUMDk6MTA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOToxNTowMC0wNTowMCwzMDAsMC4wNjEzMzMsMC4zMDY3LEROUyxJbmZyYWVzdHJ1Y3R1cmENCkYwMTAsTElORUFfQkFTRSwxMC4xMC41MC42MCxXU1VTLUNMSUVOVC02MCwxOTguNTEuMTAwLjkwLFVQREFURS1DRE4sNTUwMDEsNDQzLDYsVENQLDB4MWIsMTcwMDAwMDAsMTQ2MDAsMTIsMywyMDI2LTEwLTE0VDA5OjEwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MTU6MDAtMDU6MDAsMzAwLDAuNDUzMzMzLDIuMjY2NyxBY3R1YWxpemFjaW9uZXMsTWFudGVuaW1pZW50bw0KRjAxMSxMSU5FQV9CQVNFLDEwLjEwLjMwLjI1LFVTRVItMDI1LDE3Mi4yMC4xMC4yMCxBUEktUFJPRCw1MTAwMiw0NDMsNixUQ1AsMHgxYiw0NjAwMDAwMCw0MTQwMCwxMiwzLDIwMjYtMTAtMTRUMDk6MjA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOToyNTowMC0wNTowMCwzMDAsMS4yMjY2NjcsNi4xMzMzLEFQSSBIVFRQUyxUcsOhZmljbyBkZSBuZWdvY2lvDQpGMDEyLExJTkVBX0JBU0UsMTAuMTAuNDAuOCxVQy1DTElFTlQtMDgsMTk4LjUxLjEwMC40MCxDT0xMQUItQ0xPVUQsNTIwMDIsMzQ3OCwxNyxVRFAsMHgwMCw2NDAwMDAwMCw1MjAwMCwxMiwzLDIwMjYtMTAtMTRUMDk6MjA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOToyNTowMC0wNTowMCwzMDAsMS43MDY2NjcsOC41MzMzLENvbGFib3JhY2nDs24gVURQLENvbXVuaWNhY2lvbmVzDQpGMDEzLExJTkVBX0JBU0UsMTAuMTAuMzAuNDQsVVNFUi0wNDQsMjAzLjAuMTEzLjgwLFdFQi1TQUFTLDUzMDAyLDQ0Myw2LFRDUCwweDFiLDMwMDAwMDAwLDI0ODAwLDEyLDMsMjAyNi0xMC0xNFQwOToyMDowMC0wNTowMCwyMDI2LTEwLTE0VDA5OjI1OjAwLTA1OjAwLDMwMCwwLjgsNC4wLE5hdmVnYWNpw7NuIEhUVFBTLFRyw6FmaWNvIGRlIHVzdWFyaW8NCkYwMTQsTElORUFfQkFTRSwxMC4xMC4zMC4yNSxVU0VSLTAyNSwxMC4xMC4xLjUzLEROUy0wMSw1NDAwMiw1MywxNyxVRFAsMHgwMCwyNDAwMDAwLDg5MDAsMTIsOCwyMDI2LTEwLTE0VDA5OjIwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMDk6MjU6MDAtMDU6MDAsMzAwLDAuMDY0LDAuMzIsRE5TLEluZnJhZXN0cnVjdHVyYQ0KRjAxNSxMSU5FQV9CQVNFLDEwLjEwLjUwLjYwLFdTVVMtQ0xJRU5ULTYwLDE5OC41MS4xMDAuOTAsVVBEQVRFLUNETiw1NTAwMiw0NDMsNixUQ1AsMHgxYiwxODAwMDAwMCwxNTIwMCwxMiwzLDIwMjYtMTAtMTRUMDk6MjA6MDAtMDU6MDAsMjAyNi0xMC0xNFQwOToyNTowMC0wNTowMCwzMDAsMC40OCwyLjQsQWN0dWFsaXphY2lvbmVzLE1hbnRlbmltaWVudG8NCkYwMTYsSU5DSURFTlRFLDEwLjEwLjIwLjE1LEJLUC1TUlYtMDEsMTcyLjIwLjUuMzAsQ0xPVUQtU1RPUkFHRS0wMSw1MjE0NCw0NDMsNixUQ1AsMHgxYiwxODAwMDAwMDAwLDEyNDAwMDAsMTIsMywyMDI2LTEwLTE0VDEwOjAwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMTA6MTg6MDAtMDU6MDAsMTA4MCwxMy4zMzMzMzMsNjYuNjY2NyxCYWNrdXAgSFRUUFMsRmx1am8gc29zcGVjaG9zbw0KRjAxNyxJTkNJREVOVEUsMTAuMTAuMzAuMjUsVVNFUi0wMjUsMTcyLjIwLjEwLjIwLEFQSS1QUk9ELDUxMTEwLDQ0Myw2LFRDUCwweDFiLDExMDAwMDAwMCw5NjAwMCwxMiwzLDIwMjYtMTAtMTRUMTA6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoxODowMC0wNTowMCwxMDgwLDAuODE0ODE1LDQuMDc0MSxBUEkgSFRUUFMsU2VydmljaW8gYWZlY3RhZG8NCkYwMTgsSU5DSURFTlRFLDEwLjEwLjQwLjgsVUMtQ0xJRU5ULTA4LDE5OC41MS4xMDAuNDAsQ09MTEFCLUNMT1VELDUyMTEwLDM0NzgsMTcsVURQLDB4MDAsMjIwMDAwMDAwLDE4NDAwMCwxMiwzLDIwMjYtMTAtMTRUMTA6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoxODowMC0wNTowMCwxMDgwLDEuNjI5NjMsOC4xNDgxLENvbGFib3JhY2nDs24gVURQLFNlcnZpY2lvIGFmZWN0YWRvDQpGMDE5LElOQ0lERU5URSwxMC4xMC4zMC40NCxVU0VSLTA0NCwyMDMuMC4xMTMuODAsV0VCLVNBQVMsNTMxMTAsNDQzLDYsVENQLDB4MWIsMTQwMDAwMDAwLDExOTAwMCwxMiwzLDIwMjYtMTAtMTRUMTA6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoxODowMC0wNTowMCwxMDgwLDEuMDM3MDM3LDUuMTg1MixOYXZlZ2FjacOzbiBIVFRQUyxUcsOhZmljbyBjb25jdXJyZW50ZQ0KRjAyMCxJTkNJREVOVEUsMTAuMTAuNTAuNjAsV1NVUy1DTElFTlQtNjAsMTk4LjUxLjEwMC45MCxVUERBVEUtQ0ROLDU1MTEwLDQ0Myw2LFRDUCwweDFiLDMwMDAwMDAwMCwyNTIwMDAsMTIsMywyMDI2LTEwLTE0VDEwOjAwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMTA6MTg6MDAtMDU6MDAsMTA4MCwyLjIyMjIyMiwxMS4xMTExLEFjdHVhbGl6YWNpb25lcyxUcsOhZmljbyBjb25jdXJyZW50ZQ0KRjAyMSxJTkNJREVOVEUsMTAuMTAuMzAuMjUsVVNFUi0wMjUsMTAuMTAuMS41MyxETlMtMDEsNTQxMTAsNTMsMTcsVURQLDB4MDAsNTAwMDAwMCwxODAwMCwxMiw4LDIwMjYtMTAtMTRUMTA6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoxODowMC0wNTowMCwxMDgwLDAuMDM3MDM3LDAuMTg1MixETlMsSW5mcmFlc3RydWN0dXJhDQpGMDIyLElOQ0lERU5URSwxMC4xMC45OS4xMCxBRE1JTi0wMSwxNzIuMjAuMTAuMjAsQVBJLVBST0QsNTYxMTAsMjIsNixUQ1AsMHgxYiwxMDAwMDAwMCwxMDUwMCwxMiwzLDIwMjYtMTAtMTRUMTA6MDA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoxODowMC0wNTowMCwxMDgwLDAuMDc0MDc0LDAuMzcwNCxTU0ggYWRtaW5pc3RyYWNpw7NuLFNvcG9ydGUNCkYwMjMsUkVDVVBFUkFDSU9OLDEwLjEwLjMwLjI1LFVTRVItMDI1LDE3Mi4yMC4xMC4yMCxBUEktUFJPRCw1MTIwMCw0NDMsNixUQ1AsMHgxYiw0ODAwMDAwMCw0NDAwMCwxMiwzLDIwMjYtMTAtMTRUMTA6MjA6MDAtMDU6MDAsMjAyNi0xMC0xNFQxMDoyNTowMC0wNTowMCwzMDAsMS4yOCw2LjQsQVBJIEhUVFBTLFRyw6FmaWNvIGRlIG5lZ29jaW8NCkYwMjQsUkVDVVBFUkFDSU9OLDEwLjEwLjQwLjgsVUMtQ0xJRU5ULTA4LDE5OC41MS4xMDAuNDAsQ09MTEFCLUNMT1VELDUyMjAwLDM0NzgsMTcsVURQLDB4MDAsNjIwMDAwMDAsNTIwMDAsMTIsMywyMDI2LTEwLTE0VDEwOjIwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMTA6MjU6MDAtMDU6MDAsMzAwLDEuNjUzMzMzLDguMjY2NyxDb2xhYm9yYWNpw7NuIFVEUCxDb211bmljYWNpb25lcw0KRjAyNSxSRUNVUEVSQUNJT04sMTAuMTAuMzAuNDQsVVNFUi0wNDQsMjAzLjAuMTEzLjgwLFdFQi1TQUFTLDUzMjAwLDQ0Myw2LFRDUCwweDFiLDMxMDAwMDAwLDI2MDAwLDEyLDMsMjAyNi0xMC0xNFQxMDoyMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjI1OjAwLTA1OjAwLDMwMCwwLjgyNjY2Nyw0LjEzMzMsTmF2ZWdhY2nDs24gSFRUUFMsVHLDoWZpY28gZGUgdXN1YXJpbw0KRjAyNixSRUNVUEVSQUNJT04sMTAuMTAuMzAuMjUsVVNFUi0wMjUsMTAuMTAuMS41MyxETlMtMDEsNTQyMDAsNTMsMTcsVURQLDB4MDAsMjUwMDAwMCw5MjAwLDEyLDgsMjAyNi0xMC0xNFQxMDoyMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjI1OjAwLTA1OjAwLDMwMCwwLjA2NjY2NywwLjMzMzMsRE5TLEluZnJhZXN0cnVjdHVyYQ0KRjAyNyxSRUNVUEVSQUNJT04sMTAuMTAuNTAuNjAsV1NVUy1DTElFTlQtNjAsMTk4LjUxLjEwMC45MCxVUERBVEUtQ0ROLDU1MjAwLDQ0Myw2LFRDUCwweDFiLDE4MDAwMDAwLDE1MDAwLDEyLDMsMjAyNi0xMC0xNFQxMDoyMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjI1OjAwLTA1OjAwLDMwMCwwLjQ4LDIuNCxBY3R1YWxpemFjaW9uZXMsTWFudGVuaW1pZW50bw0KRjAyOCxSRUNVUEVSQUNJT04sMTAuMTAuMzAuMjUsVVNFUi0wMjUsMTcyLjIwLjEwLjIwLEFQSS1QUk9ELDUxMjAxLDQ0Myw2LFRDUCwweDFiLDUwMDAwMDAwLDQ1MDAwLDEyLDMsMjAyNi0xMC0xNFQxMDozMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjM1OjAwLTA1OjAwLDMwMCwxLjMzMzMzMyw2LjY2NjcsQVBJIEhUVFBTLFRyw6FmaWNvIGRlIG5lZ29jaW8NCkYwMjksUkVDVVBFUkFDSU9OLDEwLjEwLjQwLjgsVUMtQ0xJRU5ULTA4LDE5OC41MS4xMDAuNDAsQ09MTEFCLUNMT1VELDUyMjAxLDM0NzgsMTcsVURQLDB4MDAsNjQwMDAwMDAsNTMwMDAsMTIsMywyMDI2LTEwLTE0VDEwOjMwOjAwLTA1OjAwLDIwMjYtMTAtMTRUMTA6MzU6MDAtMDU6MDAsMzAwLDEuNzA2NjY3LDguNTMzMyxDb2xhYm9yYWNpw7NuIFVEUCxDb211bmljYWNpb25lcw0KRjAzMCxSRUNVUEVSQUNJT04sMTAuMTAuMzAuNDQsVVNFUi0wNDQsMjAzLjAuMTEzLjgwLFdFQi1TQUFTLDUzMjAxLDQ0Myw2LFRDUCwweDFiLDMyMDAwMDAwLDI2NzAwLDEyLDMsMjAyNi0xMC0xNFQxMDozMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjM1OjAwLTA1OjAwLDMwMCwwLjg1MzMzMyw0LjI2NjcsTmF2ZWdhY2nDs24gSFRUUFMsVHLDoWZpY28gZGUgdXN1YXJpbw0KRjAzMSxSRUNVUEVSQUNJT04sMTAuMTAuMzAuMjUsVVNFUi0wMjUsMTAuMTAuMS41MyxETlMtMDEsNTQyMDEsNTMsMTcsVURQLDB4MDAsMjYwMDAwMCw5NDAwLDEyLDgsMjAyNi0xMC0xNFQxMDozMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjM1OjAwLTA1OjAwLDMwMCwwLjA2OTMzMywwLjM0NjcsRE5TLEluZnJhZXN0cnVjdHVyYQ0KRjAzMixSRUNVUEVSQUNJT04sMTAuMTAuNTAuNjAsV1NVUy1DTElFTlQtNjAsMTk4LjUxLjEwMC45MCxVUERBVEUtQ0ROLDU1MjAxLDQ0Myw2LFRDUCwweDFiLDE5MDAwMDAwLDE1NTAwLDEyLDMsMjAyNi0xMC0xNFQxMDozMDowMC0wNTowMCwyMDI2LTEwLTE0VDEwOjM1OjAwLTA1OjAwLDMwMCwwLjUwNjY2NywyLjUzMzMsQWN0dWFsaXphY2lvbmVzLE1hbnRlbmltaWVudG8NCg=='
Path(PCAP).write_bytes(base64.b64decode(PCAP_B64))
Path(CSV).write_bytes(base64.b64decode(CSV_B64))
print('Archivos creados automáticamente:')
print('-', PCAP, Path(PCAP).stat().st_size, 'bytes')
print('-', CSV, Path(CSV).stat().st_size, 'bytes')


### Descargar el PCAP para Wireshark

    Ejecute la siguiente celda si desea descargar el PCAP y el CSV desde Colab. La descarga es opcional para continuar el análisis dentro del notebook.


In [ ]:
try:
    from google.colab import files
    files.download(PCAP)
    files.download(CSV)
except ImportError:
    print('Ejecución local: los archivos ya están en', Path.cwd())


## 2 Analizar el PCAP NetFlow v9

    El siguiente parser educativo interpreta Ethernet, IPv4, UDP, el encabezado NetFlow v9, el Template FlowSet y los Data FlowSets del archivo del taller. Está deliberadamente limitado a este dataset y no sustituye a un colector de producción.


In [ ]:
FIELD_NAMES = {1:'bytes', 2:'packets', 4:'protocol', 6:'tcp_flags', 7:'src_port',
               8:'src_ip', 10:'input_if', 11:'dst_port', 12:'dst_ip', 14:'output_if',
               21:'last_switched_ms', 22:'first_switched_ms'}

def decode_value(field_type, raw):
    if field_type in (8, 12): return socket.inet_ntoa(raw)
    return int.from_bytes(raw, 'big')

def parse_netflow_v9_pcap(filename):
    raw = Path(filename).read_bytes()
    if raw[:4] != bytes.fromhex('d4c3b2a1'):
        raise ValueError('Se esperaba PCAP little-endian clásico')
    pos, rows, templates = 24, [], {}
    while pos < len(raw):
        ts_sec, ts_usec, incl_len, _ = struct.unpack_from('<IIII', raw, pos); pos += 16
        frame = raw[pos:pos+incl_len]; pos += incl_len
        ip_offset = 14
        ihl = (frame[ip_offset] & 0x0f) * 4
        udp_offset = ip_offset + ihl
        src_exporter = socket.inet_ntoa(frame[ip_offset+12:ip_offset+16])
        dst_collector = socket.inet_ntoa(frame[ip_offset+16:ip_offset+20])
        payload = frame[udp_offset+8:]
        version, count, sys_uptime, unix_secs, sequence, source_id = struct.unpack_from('!HHIIII', payload, 0)
        if version != 9: continue
        p = 20
        while p + 4 <= len(payload):
            flowset_id, flowset_len = struct.unpack_from('!HH', payload, p)
            if flowset_len < 4 or p + flowset_len > len(payload): break
            body = payload[p+4:p+flowset_len]
            if flowset_id == 0:
                template_id, field_count = struct.unpack_from('!HH', body, 0)
                templates[template_id] = [struct.unpack_from('!HH', body, 4+i*4) for i in range(field_count)]
            elif flowset_id in templates:
                spec = templates[flowset_id]
                record_len = sum(length for _, length in spec)
                q = 0
                while q + record_len <= len(body):
                    row = {'exporter':src_exporter, 'collector':dst_collector,
                           'export_time':pd.to_datetime(unix_secs, unit='s', utc=True),
                           'sequence':sequence, 'source_id':source_id}
                    off = q
                    for field_type, length in spec:
                        chunk = body[off:off+length]; off += length
                        row[FIELD_NAMES.get(field_type, f'field_{field_type}')] = decode_value(field_type, chunk)
                    boot = row['export_time'] - pd.to_timedelta(sys_uptime, unit='ms')
                    row['start_time'] = boot + pd.to_timedelta(row['first_switched_ms'], unit='ms')
                    row['end_time'] = boot + pd.to_timedelta(row['last_switched_ms'], unit='ms')
                    rows.append(row)
                    q += record_len
            p += flowset_len
    return pd.DataFrame(rows)

pcap_df = parse_netflow_v9_pcap(PCAP)
print('Registros extraídos:', len(pcap_df))
display(pcap_df.head())


## 3 Validar la evidencia de 1.8 GB

    En NetFlow el volumen aparece como **octetos**. El valor 1 800 000 000 equivale a 1.8 GB decimales o aproximadamente 1.676 GiB.


In [ ]:
pcap_df['duration_s'] = (pcap_df['end_time'] - pcap_df['start_time']).dt.total_seconds().clip(lower=1)
pcap_df['avg_mbps'] = pcap_df['bytes'] * 8 / pcap_df['duration_s'] / 1_000_000
pcap_df['GB_decimal'] = pcap_df['bytes'] / 1_000_000_000
pcap_df['GiB_binary'] = pcap_df['bytes'] / 1024**3
top = pcap_df.nlargest(10, 'bytes')[['src_ip','dst_ip','dst_port','bytes','GB_decimal','GiB_binary','duration_s','avg_mbps']]
display(top)


**Pregunta 1.** ¿Qué campo confirma los 1.8 GB?  
    **Pregunta 2.** ¿Cuánto ancho de banda promedio consumió ese flujo?  
    **Pregunta 3.** ¿Es suficiente por sí solo para saturar 20 Mbps?


## 4 Enriquecer con inventario y contexto operativo


In [ ]:
catalog = pd.read_csv(CSV)
catalog['start_time'] = pd.to_datetime(catalog['start_time'], utc=True)
catalog['end_time'] = pd.to_datetime(catalog['end_time'], utc=True)
enriched = pcap_df.merge(catalog[['src_ip','dst_ip','src_port','dst_port','bytes','start_time',
                                  'src_name','dst_name','phase','application','expected_role']],
                         on=['src_ip','dst_ip','src_port','dst_port','bytes','start_time'], how='left')
display(enriched[['phase','src_name','dst_name','application','bytes','avg_mbps']].head(10))


## 5 Top talkers y utilización estimada


In [ ]:
top_sources = enriched.groupby(['src_ip','src_name'], dropna=False)['bytes'].sum().sort_values(ascending=False).head(10)
display((top_sources/1e9).rename('GB'))

window_load = enriched.groupby(['phase','start_time'], dropna=False)['avg_mbps'].sum()
phase_load = window_load.groupby('phase').mean().reindex(['LINEA_BASE','INCIDENTE','RECUPERACION'])
phase_pct = phase_load / LINK_MBPS * 100
summary = pd.DataFrame({'Mbps_estimados':phase_load, 'utilizacion_pct':phase_pct})
display(summary.round(2))

ax = summary['utilizacion_pct'].plot(kind='bar', color=['#66A3D2','#D95F59','#57A773'], figsize=(9,4))
ax.axhline(80, color='#D98E04', linestyle='--', label='Umbral 80%')
ax.axhline(100, color='#A61B1B', linestyle='--', label='Capacidad 100%')
ax.set_ylabel('% estimado del enlace')
ax.set_title('Comparación de carga por fase')
ax.legend(); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 6 Detección simple de anomalías

    Se compara cada flujo con la mediana de bytes de la línea base. Es una técnica educativa; en producción se deben separar perfiles por aplicación, horario e interfaz.


In [ ]:
baseline = enriched.loc[enriched.phase.eq('LINEA_BASE'), 'bytes']
median = baseline.median()
mad = (baseline - median).abs().median()
enriched['robust_z'] = 0.6745 * (enriched['bytes'] - median) / max(mad, 1)
anomalies = enriched.loc[enriched.robust_z.abs() > 3.5,
    ['phase','src_name','dst_name','application','bytes','avg_mbps','robust_z']].sort_values('robust_z', ascending=False)
display(anomalies)


## 7 Conclusión técnica guiada

    Complete antes de revisar la solución:

    1. Host origen que generó el mayor volumen.
    2. Destino, puerto y protocolo.
    3. Volumen decimal y binario.
    4. Tasa promedio del flujo principal.
    5. Utilización combinada durante el incidente.
    6. Evidencia que permite diferenciar correlación de causa raíz.
    7. Acción inmediata y control preventivo recomendado.


## 8 Solución esperada

    - `BKP-SRV-01` (`10.10.20.15`) transfirió 1 800 000 000 octetos hacia `172.20.5.30:443`.
    - Ese valor corresponde a **1.8 GB decimales** o **1.676 GiB**.
    - Duración: 1 080 segundos. Promedio: aproximadamente **13.33 Mbps**, equivalente a 66.7% del enlace.
    - Sumado al tráfico concurrente, la estimación supera aproximadamente el 95% del enlace de 20 Mbps.
    - NetFlow prueba volumen, actores, periodo y puerto, pero no prueba por sí solo pérdida, latencia ni errores de aplicación. Para confirmar el impacto deben correlacionarse SNMP, logs, métricas o trazas.
    - Contención: pausar o limitar el respaldo. Prevención: ventana de backup, QoS/policing, alerta de top talker y umbral de utilización.


## 9 Exportar evidencia


In [ ]:
evidence = enriched.sort_values('bytes', ascending=False).head(10)
evidence.to_csv('evidencia_top_talkers.csv', index=False)
print('Generado: evidencia_top_talkers.csv')
try:
    from google.colab import files
    files.download('evidencia_top_talkers.csv')
except ImportError:
    pass
